In [1]:
import pandas as pd
import re
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
# Your CSV path on Windows
file_path = r"C:\Users\16248\Desktop\大学课程\541 Practical Code\summaries_output_V3__01_15_2025.csv"

df = pd.read_csv(file_path)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (80, 7)
Columns: ['model', 'title', 'side', 'article_url', 'original_text_alternating_op', 'prompt', 'output']


,model,title,side,article_url,original_text_alternating_op,prompt,output
0,llama4-maverick,1st 2020 Democratic Debate,left,https:/theflipside.io/archives/1st-2020-democr...,"Regarding the debate questions, ""It's hard to ...",neutral,The recent Democratic presidential debate feat...
1,llama4-maverick,1st 2020 Democratic Debate,left,https:/theflipside.io/archives/1st-2020-democr...,"Regarding the debate questions, ""It's hard to ...",left,Here is a summary of the articles from a left-...
2,llama4-maverick,1st 2020 Democratic Debate,left,https:/theflipside.io/archives/1st-2020-democr...,"Regarding the debate questions, ""It's hard to ...",right,Here is a summary of the article from a right-...
3,llama4-maverick,1st 2020 Democratic Debate,left,https:/theflipside.io/archives/1st-2020-democr...,"Regarding the debate questions, ""It's hard to ...",identify,Democratic Primary Debate
4,deepseek-v3p1,1st 2020 Democratic Debate,left,https:/theflipside.io/archives/1st-2020-democr...,"Regarding the debate questions, ""It's hard to ...",neutral,A collection of post-debate commentary offers ...


In [3]:
# Keep only rows that do NOT contain "identify" in the prompt
df = df[~df["prompt"].astype(str).str.lower().str.contains("identify", na=False)].copy()

print("Shape after removing 'identify' rows:", df.shape)
df.head()

Shape after removing 'identify' rows: (60, 7)


,model,title,side,article_url,original_text_alternating_op,prompt,output
0,llama4-maverick,1st 2020 Democratic Debate,left,https:/theflipside.io/archives/1st-2020-democr...,"Regarding the debate questions, ""It's hard to ...",neutral,The recent Democratic presidential debate feat...
1,llama4-maverick,1st 2020 Democratic Debate,left,https:/theflipside.io/archives/1st-2020-democr...,"Regarding the debate questions, ""It's hard to ...",left,Here is a summary of the articles from a left-...
2,llama4-maverick,1st 2020 Democratic Debate,left,https:/theflipside.io/archives/1st-2020-democr...,"Regarding the debate questions, ""It's hard to ...",right,Here is a summary of the article from a right-...
4,deepseek-v3p1,1st 2020 Democratic Debate,left,https:/theflipside.io/archives/1st-2020-democr...,"Regarding the debate questions, ""It's hard to ...",neutral,A collection of post-debate commentary offers ...
5,deepseek-v3p1,1st 2020 Democratic Debate,left,https:/theflipside.io/archives/1st-2020-democr...,"Regarding the debate questions, ""It's hard to ...",left,"From a left-leaning perspective, the debate is..."


In [4]:
def extract_true_label(prompt_text):
    """
    Extract the gold stance label from the prompt column.
    """
    if pd.isna(prompt_text):
        return None
    
    text = str(prompt_text).lower()
    
    if "favor" in text:
        return "favor"
    elif "against" in text:
        return "against"
    elif "neither" in text or "neutral" in text:
        return "neither"
    else:
        return None

In [5]:
def predict_stance_keyphrase(output_text):
    """
    A simple T5 keyphrase-style stance predictor using keyword scoring.
    This is not a real fine-tuned T5 model, but a notebook-friendly approximation.
    """
    if pd.isna(output_text):
        return None
    
    text = str(output_text).lower()

    favor_keywords = [
        "support", "supports", "supporting",
        "benefit", "benefits", "beneficial",
        "positive", "advantage", "advantages",
        "help", "helps", "helpful",
        "improve", "improves", "improved",
        "good", "useful", "effective", "encourage"
    ]
    
    against_keywords = [
        "oppose", "opposes", "opposing",
        "harm", "harms", "harmful",
        "bad", "negative",
        "risk", "risks",
        "problem", "problems",
        "damage", "damages",
        "hurt", "hurts",
        "danger", "dangerous",
        "criticize", "criticizes"
    ]
    
    neither_keywords = [
        "neutral", "balanced", "mixed", "unclear",
        "both sides", "depends", "complex", "uncertain"
    ]

    favor_score = sum(1 for kw in favor_keywords if kw in text)
    against_score = sum(1 for kw in against_keywords if kw in text)
    neither_score = sum(1 for kw in neither_keywords if kw in text)

    if favor_score > against_score and favor_score >= neither_score:
        return "favor"
    elif against_score > favor_score and against_score >= neither_score:
        return "against"
    else:
        return "neither"

In [6]:
# Apply label extraction and prediction
df["true_label"] = df["prompt"].apply(extract_true_label)
df["predicted_label"] = df["output"].apply(predict_stance_keyphrase)

# Keep only rows where a gold label was found
result_df = df.dropna(subset=["true_label", "predicted_label"]).copy()

print("Rows used for evaluation:", len(result_df))
result_df[["prompt", "output", "true_label", "predicted_label"]].head(10)

Rows used for evaluation: 20


,prompt,output,true_label,predicted_label
0,neutral,The recent Democratic presidential debate feat...,neither,neither
4,neutral,A collection of post-debate commentary offers ...,neither,neither
8,neutral,The first debate among Democratic presidential...,neither,neither
12,neutral,The first Democratic presidential debate featu...,neither,neither
16,neutral,The inclusion of a citizenship question on the...,neither,neither
20,neutral,A legal and political debate exists over the a...,neither,against
24,neutral,The inclusion of a citizenship question on the...,neither,neither
28,neutral,The debate over including a citizenship questi...,neither,favor
32,neutral,The 2020 Democratic presidential primary is sh...,neither,neither
36,neutral,The text argues that the Democratic Party is m...,neither,neither


In [7]:
# Accuracy
acc = accuracy_score(result_df["true_label"], result_df["predicted_label"])
print("Accuracy:", round(acc, 4))

Accuracy: 0.75


In [8]:
# Detailed report
print(classification_report(
    result_df["true_label"],
    result_df["predicted_label"],
    labels=["favor", "against", "neither"],
    zero_division=0
))

              precision    recall  f1-score   support

       favor       0.00      0.00      0.00         0
     against       0.00      0.00      0.00         0
     neither       1.00      0.75      0.86        20

    accuracy                           0.75        20
   macro avg       0.33      0.25      0.29        20
weighted avg       1.00      0.75      0.86        20



In [9]:
# Confusion matrix
cm = confusion_matrix(
    result_df["true_label"],
    result_df["predicted_label"],
    labels=["favor", "against", "neither"]
)

cm_df = pd.DataFrame(
    cm,
    index=["true_favor", "true_against", "true_neither"],
    columns=["pred_favor", "pred_against", "pred_neither"]
)

cm_df

,pred_favor,pred_against,pred_neither
true_favor,0,0,0
true_against,0,0,0
true_neither,3,2,15


In [10]:
# Save the row-by-row predictions
output_file = r"C:\Users\16248\Desktop\大学课程\541 Practical Code\t5_keyphrase_results.csv"
result_df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("Saved results to:")
print(output_file)

Saved results to:
C:\Users\16248\Desktop\大学课程\541 Practical Code\t5_keyphrase_results.csv
